# types

> Normalized request/response dataclasses shared across providers.

In [ ]:
#| default_exp types


## Notebook Overview

This notebook documents `fastllm_v2.types` in nbdev style, interleaving explanations, source cells, and lightweight REPL checks.

- **Depends on:** none (leaf module)
- **Primary symbols:** `Part, Msg, ToolCall, Usage, Caps, RequestOptions, Completion, Delta`


## Module Setup

Imports, constants, and shared setup used by the symbols below.


In [ ]:
#| export
"Core internal types."

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional


### `Part`

A normalized content part.


In [ ]:
#| export


@dataclass(frozen=True)
class Part:
    "A normalized content part."
    type: str
    text: Optional[str] = None
    data: Optional[Dict[str, Any]] = None


### `Msg`

A normalized message.


In [ ]:
#| export


@dataclass(frozen=True)
class Msg:
    "A normalized message."
    role: str
    content: List[Part]
    data: Optional[Dict[str, Any]] = None


### `ToolCall`

Normalized tool call.


In [ ]:
#| export


@dataclass(frozen=True)
class ToolCall:
    "Normalized tool call."
    id: str
    name: str
    arguments: Dict[str, Any] = field(default_factory=dict)


### `Usage`

Normalized usage.


In [ ]:
#| export


@dataclass(frozen=True)
class Usage:
    "Normalized usage."
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    raw: Dict[str, Any] = field(default_factory=dict)


### `Caps`

Capability declaration for a client.


In [ ]:
#| export


@dataclass(frozen=True)
class Caps:
    "Capability declaration for a client."
    tools: bool = False
    tool_choice: bool = False
    streaming: bool = True
    search: bool = False
    reasoning: bool = False
    prefill: bool = False
    citations: bool = False
    prompt_caching: bool = False
    images: bool = True
    pdfs: bool = False
    url_context: bool = False


### `RequestOptions`

Request options shared across providers.


In [ ]:
#| export


@dataclass(frozen=True)
class RequestOptions:
    "Request options shared across providers."
    max_tokens: Optional[int] = None
    temperature: Optional[float] = None
    cache: Optional[Any] = None
    tools: Optional[List[Dict[str, Any]]] = None
    tool_choice: Optional[Any] = None
    reasoning_effort: Optional[str] = None
    response_format: Optional[Dict[str, Any]] = None
    native: Optional[Dict[str, Any]] = None
    extra_body: Optional[Dict[str, Any]] = None
    extra_headers: Optional[Dict[str, str]] = None
    extra_query: Optional[Dict[str, Any]] = None


### `Completion`

Normalized completion response.


In [ ]:
#| export


@dataclass(frozen=True)
class Completion:
    "Normalized completion response."
    model: str
    message: Msg
    finish_reason: Optional[str] = None
    usage: Optional[Usage] = None
    tool_calls: List[ToolCall] = field(default_factory=list)
    provider: Optional[str] = None
    raw: Dict[str, Any] = field(default_factory=dict)


### `Delta`

Normalized streaming delta event.


In [ ]:
#| export


@dataclass(frozen=True)
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    tool_calls: List[ToolCall] = field(default_factory=list)
    finish_reason: Optional[str] = None
    usage: Optional[Usage] = None
    raw: Dict[str, Any] = field(default_factory=dict)


## Field Reference

This section is a practical reference for what each dataclass field can contain in real usage.
The values below are based on how `fastllm_v2` currently coerces inputs and serializes payloads.


### `Part` field values

| Field | Type | Common values | Notes |
|---|---|---|---|
| `type` | `str` | `text`, `image`/`image_url`/`input_image`, `audio`/`input_audio`, `file`/`input_file`/`pdf`/`document`, `tool_use` | Unknown values are passed through as provider-native part types. |
| `text` | `Optional[str]` | plain text, URL, or `None` | For media parts, `text` can be used as a URL fallback when `data` does not include it. |
| `data` | `Optional[dict]` | provider payload dict or `None` | Raw provider-specific fields; merged into serialized payload when present. |

Typical `Part` examples:
```python
Part(type='text', text='Explain SSE briefly.')
Part(type='image_url', text='https://.../cat.png')
Part(type='input_image', data={'image_url':'https://.../cat.png', 'detail':'high'})
Part(type='input_file', data={'file_id':'file_123'})
Part(type='tool_use', data={'id':'toolu_1','name':'web_search','input':{'query':'istanbul weather'}})
```


### `Msg` field values

| Field | Type | Common values | Notes |
|---|---|---|---|
| `role` | `str` | `system`, `user`, `assistant`, `tool` | `highlevel._coerce_msg` defaults missing role to `user`. |
| `content` | `list[Part]` | one or many `Part` objects | Non-assistant empty content is coerced to `[Part(type='text', text='')]` by high-level helpers. |
| `data` | `Optional[dict]` | metadata or provider override blocks | Provider-native shortcuts can be embedded under keys like `openai`, `openai_chat`, `anthropic`, `gemini`. |

Examples:
```python
Msg(role='user', content=[Part(type='text', text='Hello')])
Msg(role='assistant', content=[Part(type='text', text='Hi there')])
Msg(role='user', content=[Part(type='text', text='Describe this image'), Part(type='image_url', text='https://...')])
```


### `ToolCall` and `Usage` field values

`ToolCall`:
- `id: str`: provider tool call id (for example `call_abc`, `toolu_...`) or synthesized stream key.
- `name: str`: function/tool name. During streaming this may temporarily be empty until later deltas arrive.
- `arguments: dict`: parsed JSON args when available. Partial streams may use helper keys like `_delta` / `_raw`.

`Usage`:
- `prompt_tokens`, `completion_tokens`, `total_tokens`: normalized integer counters (default `0`).
- `raw`: provider-native usage block (`input_tokens`, `cachedContentTokenCount`, etc.).

Examples:
```python
ToolCall(id='call_1', name='simple_add', arguments={'a': 2, 'b': 3})
ToolCall(id='call_2', name='', arguments={'_delta': '{"a":2'})

Usage(prompt_tokens=12, completion_tokens=5, total_tokens=17)
Usage(prompt_tokens=1401, completion_tokens=2, total_tokens=1438,
      raw={'cachedContentTokenCount': 1008})
```


### `Caps` field values

All fields are booleans describing client capabilities:
- `tools`, `tool_choice`, `streaming`, `search`, `reasoning`, `prefill`, `citations`, `prompt_caching`, `images`, `pdfs`, `url_context`

Example:
```python
Caps(tools=True, tool_choice=True, streaming=True, search=True,
     reasoning=True, prompt_caching=True, images=True, pdfs=False)
```


### `RequestOptions` field values

| Field | Type | Typical values |
|---|---|---|
| `max_tokens` | `Optional[int]` | positive ints like `256`, `1024` |
| `temperature` | `Optional[float]` | `0.0` to around `2.0` depending on provider |
| `cache` | `Optional[Any]` | `True`/`False`, provider cache dict, or Gemini `cachedContent` string |
| `tools` | `Optional[list[dict]]` | function tool schemas or provider-native tool blocks |
| `tool_choice` | `Optional[Any]` | `'auto'`, `'required'`, `'none'`, or provider dict |
| `reasoning_effort` | `Optional[str]` | `'minimal'`, `'low'`, `'medium'`, `'high'`, `'max'`, `'very_high'` |
| `response_format` | `Optional[dict]` | JSON-schema/text format objects |
| `native` | `Optional[dict]` | provider-native payload fields merged directly |
| `extra_body` | `Optional[dict]` | extra body keys merged after normalized fields |
| `extra_headers` | `Optional[dict[str,str]]` | request headers like beta flags |
| `extra_query` | `Optional[dict]` | query params for provider-specific endpoints |

Example:
```python
RequestOptions(
    max_tokens=512,
    temperature=0.2,
    cache=True,
    tools=[{'type':'function','function':{'name':'simple_add','parameters':{'type':'object'}}}],
    tool_choice='auto',
    reasoning_effort='medium',
    extra_headers={'anthropic-beta':'search-results-2025-06-09'},
)
```


### `Completion` and `Delta` field values

`Completion` is the final normalized response:
- `model: str`: resolved response model name.
- `message: Msg`: normalized assistant message.
- `finish_reason: Optional[str]`: provider-specific stop reason (examples: `stop`, `length`, `tool_calls`, `end_turn`, `STOP`, `completed`).
- `usage: Optional[Usage]`: usage block if provider returns one.
- `tool_calls: list[ToolCall]`: extracted final tool calls.
- `provider: Optional[str]`: provider family when available (`openai`, `anthropic`, `gemini`, `openai_compat`).
- `raw: dict`: full provider-native response payload.

`Delta` is one streaming event:
- `text: str`: incremental text chunk (can be empty on non-text events).
- `tool_calls: list[ToolCall]`: partial or completed tool call fragments.
- `finish_reason: Optional[str]`: usually set near the final events.
- `usage: Optional[Usage]`: commonly appears in final event(s).
- `raw: dict`: original SSE/event payload.


### End-to-end type examples

```python
from fastllm_v2.types import Part, Msg, ToolCall, Usage, Completion, Delta

msg = Msg(
    role='user',
    content=[
        Part(type='text', text='What is 2+3? Use tool.'),
        Part(type='image_url', text='https://example.com/diagram.png'),
    ],
)

tool_call = ToolCall(id='call_1', name='simple_add', arguments={'a': 2, 'b': 3})
usage = Usage(prompt_tokens=20, completion_tokens=8, total_tokens=28)

final = Completion(
    model='gpt-5-mini',
    message=Msg(role='assistant', content=[Part(type='text', text='2 + 3 = 5')]),
    finish_reason='stop',
    usage=usage,
    tool_calls=[tool_call],
    provider='openai',
)

stream_ev = Delta(text='2 + 3', tool_calls=[], finish_reason=None, usage=None)
```


In [ ]:
# example: quick object construction sanity check
from fastcore.test import test_eq
from fastllm_v2.types import Part, Msg, ToolCall, Usage, Completion, Delta, RequestOptions

m = Msg(role='user', content=[Part(type='text', text='hello')])
tc = ToolCall(id='call_1', name='simple_add', arguments={'a': 1, 'b': 2})
u = Usage(prompt_tokens=10, completion_tokens=4, total_tokens=14)
res = Completion(model='demo-model', message=Msg(role='assistant', content=[Part(type='text', text='3')]), usage=u, tool_calls=[tc])
d = Delta(text='3', usage=u)
opts = RequestOptions(max_tokens=128, temperature=0.1, cache=True)

test_eq(m.role, 'user')
test_eq(tc.arguments['a'] + tc.arguments['b'], 3)
test_eq(res.usage.total_tokens, 14)
test_eq(d.text, '3')
test_eq(opts.max_tokens, 128)


## REPL Checks

Quick smoke checks to confirm exported symbols import correctly.


In [ ]:
#| hide
from fastcore.test import test
import fastllm_v2.types as m

test(m is not None)
test(hasattr(m, 'Part'))
